# Stage 3, step 1: the BioLangFusion benchmark datasets

BioLangFusion combines features from DNA, RNA, and protein encoders to predict biological properties. Before interpreting a model comparison, we need to know what its examples represent, which examples were used, and how training and test data were separated. This notebook checks those prerequisites for the project's reproduction.

The research plan's Priority 1, Phase A aims to reproduce the paper's encoders, codon correspondence, three fusion methods, TextCNN prediction head, and evaluation on five benchmarks ([Gantt and Details, task 17](../../docs/Research_Project_Plan_Gantt.xlsx)). A documented adaptation can still be informative, but it must be distinguished from an exact reproduction. Matching a score alone would not establish that the procedures match.

The reading path follows the investigation:

1. **Published reference:** which datasets, scores, and sequence counts does the paper report?
2. **Available files:** what do our sequences, targets, and split columns contain?
3. **Preprocessing:** how does our length filter compare with the paper's description and counts?
4. **Evaluation and sequence checks:** do repeated sequences cross split boundaries, and what assumptions are needed to construct protein inputs?
5. **Handoff:** what can the embedding step use, and what remains unresolved?

This notebook loads no pretrained models, computes no embeddings, and writes no dataset files. [Step 2](02_token_embeddings.ipynb) extracts embeddings; [step 3](03_blf_build_check.ipynb) builds and checks the fusion models. The [Stage 3 overview](README.md) locates those steps. The [literature review](<../../docs/Literature Review - Biological Foundation Models - Comparative Draft v5.pdf>), especially “From Biological Sequences to Model Predictions,” provides optional biological background; the explanations needed for each check remain here.

**Reading the saved results.** Outputs below are retained observations from earlier executions, not a fresh run of this revision. The published-score table also predates an earlier source correction and is marked beside its output. Current code, published claims, saved observations, and unresolved interpretations are distinguished throughout; this audit itself does not demonstrate successful model reproduction.

---

## Setup

The audit uses pandas and the project's dataset registry. The registry in [`Code/shared_code/datasets.py`](../shared_code/datasets.py) identifies each dataset's file, sequence column, target column, task, and split column. Using it keeps this notebook consistent with the later pipeline.

The setup separates ordinary imports, locating the shared package, and choosing the registered benchmarks. Importing the registry does not load the CSVs or pretrained models.

In [15]:
# Standard library
import sys
from pathlib import Path

import pandas as pd

In [16]:
# Find Code/ from this notebook's working directory or its parents.
# From the repository root, the fallback is the Code/ child directory.
CODE_DIR = Path.cwd() / "Code"
for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "shared_code").is_dir():
        CODE_DIR = folder
        break

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Loads Code/shared_code/datasets.py: the registry and dataset-loading helpers.
from shared_code import datasets

In [17]:
pd.set_option("display.width", 120)

# Four registered benchmarks; the paper's fifth dataset, Ab1, is still missing.
BENCHMARKS = datasets.BLF_BENCHMARKS
print("Registered BioLangFusion benchmarks:", ", ".join(BENCHMARKS))
print("Length rule applied by blf_benchmark_rows:", datasets.BLF_LENGTH_LIMIT, "nucleotides")

Registered BioLangFusion benchmarks: fungal_expression, ecoli_proteins, mrna_stability, cov_vac_degradation
Length rule applied by blf_benchmark_rows: 999 nucleotides


`sys.path` tells Python where to find the project's `shared_code` package; `datasets` is the module imported from that package. The CSVs are resolved separately through `datasets.DEFAULT_DATA_DIR`, which points to the repository's `datasets/` folder.

The printed **999 nucleotides** is the shared loader's current cutoff, not a conclusion established by importing it. We examine its evidence below. These checks perform no random sampling or model fitting, so they need no seed; training reproducibility belongs to the later experiments.

---

## The five benchmarks BioLangFusion reports

A benchmark combines a dataset, a prediction task, and an evaluation procedure. Here the targets concern expression, mRNA stability, or degradation. Expression concerns the production or abundance of RNA or protein; mRNA stability concerns persistence, while degradation concerns breakdown. These terms give biological context, but the exact assay and units require dataset provenance.

An encoder converts a sequence into numerical features; a fusion method combines features from different encoders before prediction.

**Why compare with the strongest single encoder?** Fusion should demonstrate value beyond a strong individual representation. For each task, we therefore compare the highest single-encoder score with the highest score among the three main fusion methods in the paper's Table 1. This is a selected comparison, not a claim that every fusion method improves every task or that the same method wins throughout.

The encoders used for fusion are **Nucleotide Transformer v2 100M multi-species** (DNA), **RNA-FM** (RNA), and **ESM-2 8M** (protein). Our implementation uses frozen encoders—their pretrained weights remain unchanged—then trains downstream fusion and prediction components. These streams are derived from the supplied sequence; they are not three independently measured molecular assays.

The table transcribes scores and fusion methods from [BioLangFusion, Table 1](../../Papers/core/BioLangFusion.pdf), sequence counts from Table 2, and uses the model identities in Table 3. `best head` is the existing column name for the selected **fusion method**, not the downstream prediction head. Appendix ablations are outside this summary.

**Why include counts?** The `used` column records how many sequences the paper reports using. Later checks compare those counts with our files and filtering. Count agreement is a useful consistency check, but cannot establish that the examples or splits match.

In [18]:
# BioLangFusion Table 1: scores and best-performing fusion method.
# BioLangFusion Table 2: number of sequences used.
# Registry keys connect these paper benchmarks to shared_code/datasets.py.
PUBLISHED = pd.DataFrame(
    [
        ("Fungal expression",   "fungal_expression",   "Spearman",     3138, 0.804,  0.767,  0.695,  0.824,  "MIL + entropy"),
        ("E. coli proteins",    "ecoli_proteins",      "accuracy (%)", 4450, 41.292, 52.949, 49.017, 53.932, "cross-attention"),
        ("mRNA stability",      "mrna_stability",      "Spearman",    23929, 0.530,  0.553,  0.539,  0.563,  "MIL + entropy"),
        ("Ab1 antibody mRNA",   None,                  "Spearman",      723, 0.732,  0.743,  0.711,  0.765,  "cross-attention"),
        ("CoV-Vac degradation", "cov_vac_degradation", "Spearman",     2400, 0.780,  0.841,  0.806,  0.864,  "MIL + entropy"),
    ],
    columns=[
        "benchmark",
        "registry key",
        "metric",
        "used",
        "NT",
        "RNA-FM",
        "ESM-2 8M",
        "best fused",
        "best head",
    ],
)

In [19]:
# Find the highest single-encoder score within each benchmark row.
single_encoder_columns = ["NT", "RNA-FM", "ESM-2 8M"]
PUBLISHED["best single"] = PUBLISHED[single_encoder_columns].max(axis=1)

# Calculate the score difference in each benchmark's metric units.
PUBLISHED["fusion gain"] = (
    PUBLISHED["best fused"] - PUBLISHED["best single"]
).round(3)

# Check registry membership; this does not check whether files are available.
PUBLISHED["registered"] = (
    PUBLISHED["registry key"]
    .isin(datasets.DATASETS)
    .map({True: "yes", False: "no"})
)

# Use benchmark names as row labels for display.
PUBLISHED.set_index("benchmark")[
    [
        "metric",
        "used",
        "registered",
        "NT",
        "RNA-FM",
        "ESM-2 8M",
        "best single",
        "best fused",
        "best head",
        "fusion gain",
    ]
]

,metric,used,registered,NT,RNA-FM,ESM-2 8M,best single,best fused,best head,fusion gain
benchmark,,,,,,,,,,
Fungal expression,Spearman,3138,yes,0.804,0.767,0.695,0.804,0.824,MIL + entropy,0.020
E. coli proteins,accuracy (%),4450,yes,41.292,52.949,49.017,52.949,53.932,cross-attention,0.983
mRNA stability,Spearman,23929,yes,0.530,0.553,0.539,0.553,0.563,MIL + entropy,0.010
Ab1 antibody mRNA,Spearman,723,no,0.732,0.743,0.711,0.743,0.765,cross-attention,0.022
CoV-Vac degradation,Spearman,2400,yes,0.780,0.841,0.806,0.841,0.864,MIL + entropy,0.023


**Saved output.** This table has not been regenerated since the earlier code correction. It still shows rounded E. coli scores, `in hand` instead of `registered`, and `gated MIL` instead of the paper's `MIL + entropy` label. Running the two preceding cells would refresh it; no run is implied by the revised explanation.

`max(axis=1)` compares the three single-encoder scores within each row. The next line calculates:

$$
\text{fusion gain} = \text{best reported fusion score} - \text{best reported single-encoder score}.
$$

For fungal expression, this is $0.824 - 0.804 = 0.020$. Across the four regression tasks, the differences are **0.010–0.023 in Spearman correlation**, which measures agreement between predicted and observed rankings rather than the accuracy of their numerical values. E. coli uses classification accuracy: $53.932 - 52.949 = 0.983$ **percentage points**. Do not average differences expressed in these different units.

Table 1 supplies point estimates without uncertainty estimates. The planned repeated-seed comparisons and paired differences—comparing fusion with its baseline under matched evaluation conditions for each repeat—address consistency in our reproduction. The published differences alone establish neither statistical reliability nor a biological mechanism.

`registered` checks membership in `datasets.DATASETS`. It does not establish file availability or data quality; the next section inspects the registered files.

### The missing fifth benchmark: Ab1

BioLangFusion reports **723 antibody-encoding mRNA sequences**, a maximum length of **1,203 nucleotides**, and protein-expression targets for Ab1. Appendix A.2 attributes it to **Yazdani et al.**, separately from the four CodonBERT benchmarks. Ab1 is not registered here, so the current audit covers four of the five benchmarks.

The 1,203-nucleotide maximum is a dataset statistic; our 999-nucleotide cutoff is an implemented filtering choice. They describe different things, but their relationship matters: Table 2 reports all 723 Ab1 examples retained. A uniform rule discarding every sequence above 999 nucleotides would not account for that description. The preprocessing section returns to this unresolved difference.

---

## What is in each file

Before applying the length filter, inspect the four registered files. A **row** is one recorded sequence–target observation; multiple rows can contain the same sequence. This distinction becomes important when checking repeated observations and train/test overlap.

The files share `Sequence`, `Value`, `Dataset`, and `Split` columns. The registry tells the code which columns to read and whether the task is regression or classification. `Value` is a generic target name: the column alone does not establish an assay, physical unit, or normalization procedure.

The loop reads each CSV and reports its size, columns, sequence lengths, and target range. Trimming whitespace and converting to uppercase creates a comparison view without modifying the file. The alphabet summary samples the **first 500 rows**; the later sequence checks inspect all retained rows. This is a descriptive inspection, not a complete schema or missing-value validation.

In [20]:
# Inspect the supplied CSV before the shared length filter.
rows = []
for key in BENCHMARKS:
    spec = datasets.DATASETS[key]
    # The registry supplies the file and column names.
    frame = pd.read_csv(datasets.DEFAULT_DATA_DIR / spec.path)
    seq = frame[spec.sequence_column].astype(str).str.strip().str.upper()
    value = frame[spec.label_column]
    # Alphabet is sampled from 500 rows; lengths and target summaries use the full frame.
    rows.append({
        "benchmark": spec.label,
        "file": spec.path,
        "rows": len(frame),
        "columns": ", ".join(frame.columns),
        "alphabet": "".join(sorted(set("".join(seq.head(500))))),
        "length min": int(seq.str.len().min()),
        "length median": int(seq.str.len().median()),
        "length max": int(seq.str.len().max()),
        "task": spec.task,
        "label min": round(float(value.min()), 3),
        "label max": round(float(value.max()), 3),
        "distinct labels": int(value.nunique()),
    })
AUDIT = pd.DataFrame(rows).set_index("benchmark")
AUDIT

,file,rows,columns,alphabet,length min,length median,length max,task,label min,label max,distinct labels
benchmark,,,,,,,,,,,
Fungal expression,Fungal_expression.csv,7089,"Sequence, Value, Dataset, Split",ACGU,150,1086,3063,regression,-8.977,5.14,5134
"E. coli protein expression (low, medium, high)",E.Coli_proteins.csv,6348,"Sequence, Value, Dataset, Split",ACGU,171,747,3000,classification,0.000,2.00,3
mRNA stability,mRNA_Stability.csv,65356,"Sequence, Value, Dataset, Split",ACGU,3,1239,3066,regression,-3.109,2.89,65192
CoV-Vac degradation,CoV_Vaccine_Degradation.csv,2400,"Sequence, Value, Dataset, Split",ACGU,81,81,81,regression,-7.154,6.51,2378


The saved inspection shows RNA letters in the sampled rows, three registered regression tasks with numerical targets, and the E. coli classification task with three label values. Including the unavailable Ab1 benchmark gives four regression tasks in the paper overall. The paper specifies Spearman correlation for regression and accuracy for E. coli; those metrics are choices of evaluation, not consequences of a column's data type alone.

CoV-Vac rows are fixed at **81 nucleotides**, while the other files contain sequences longer than 999 nucleotides. The project treats the CoV-Vac inputs as fragments; their exact biological coordinates and reading frame require source information. RNA letters or a length divisible by three would not, by themselves, establish a complete coding sequence.

The saved row counts exceed the paper's `used` counts for fungal expression, E. coli, and mRNA stability. This motivates the length-filter investigation below. Later, [Sequence checks](#Sequence-checks) examines whether retained inputs satisfy the assumptions needed for the codon grid.

---

## Checking BioLangFusion's preprocessing

**Question:** can a length filter on our available files reproduce the paper's retained counts, and does that agree with the stated method?

[BioLangFusion, Appendix A.2](../../Papers/core/BioLangFusion.pdf) describes **truncating** long RNA inputs around a stated 1,000-token constraint. Our [`blf_benchmark_rows`](../shared_code/datasets.py) instead **discards rows** whose sequences exceed `BLF_LENGTH_LIMIT`, currently 999 nucleotides. It also drops missing sequences, strips surrounding whitespace, converts letters to uppercase, and preserves the supplied split and file order. Its existing docstring calls this published preprocessing; the checks here establish only a count-matching implementation, not verified author behavior.

For a 1,203-nucleotide input, filtering at 999 removes the entire observation; truncating to 999 keeps an observation with a shortened sequence. Filtering changes which examples are evaluated. Truncation changes what sequence information the model sees. These are scientifically different operations.

**Nucleotide length** counts individual bases represented by sequence letters; a **token** is a unit produced by an encoder's tokenizer. One token can represent one nucleotide, a nucleotide group, or a protein residue, and model inputs may also contain framing tokens. Therefore the paper's token statement does not by itself establish a universal 999-nucleotide cutoff.

The first cell tests limits from **900 through 1,100 nucleotides** against all four published counts. The second shows the current default's retained counts beside the targets. This investigates a candidate explanation; matching counts cannot rule out other preprocessing rules.

In [21]:
# Map registry keys to the paper's used counts, then test candidate length limits.
target = PUBLISHED.dropna(subset=["registry key"]).set_index("registry key")["used"].to_dict()
matching = [
    limit for limit in range(900, 1101)
    if all(len(datasets.blf_benchmark_rows(k, length_limit=limit)) == target[k] for k in BENCHMARKS)
]

In [22]:
# Report the sweep and the current loader default (999 nt).
print("Length limits reproducing all four published counts:", matching)

SWEEP = pd.DataFrame([
    {
        "benchmark": datasets.DATASETS[k].label,
        "rows in file": len(datasets.blf_benchmark_rows(k, length_limit=10**9)),
        "kept at 999 nt": len(datasets.blf_benchmark_rows(k)),
        "published used": target[k],
        "matches": len(datasets.blf_benchmark_rows(k)) == target[k],
    }
    for k in BENCHMARKS
]).set_index("benchmark")
SWEEP

Length limits reproducing all four published counts: [999, 1000, 1001]


,rows in file,kept at 999 nt,published used,matches
benchmark,,,,
Fungal expression,7089,3138,3138,True
"E. coli protein expression (low, medium, high)",6348,4450,4450,True
mRNA stability,65356,23929,23929,True
CoV-Vac degradation,2400,2400,2400,True


**What the saved sweep establishes.** Limits of **999, 1,000, and 1,001 nucleotides** all match the four reported counts: 3,138 fungal, 4,450 E. coli, 23,929 mRNA stability, and 2,400 CoV-Vac rows. The implementation selects 999. Equal counts at these limits mean the intervening lengths do not change the retained rows in this sweep; they do not uniquely identify the authors' cutoff or explain it through a special token.

The table's `rows in file` column calls the helper with a very large length cap, so it counts rows **after the helper removes missing sequences**. The earlier raw CSV inspection is the reference for total file rows. Likewise, `kept at 999 nt` calls the helper's current default; the column label must be revisited if the shared cutoff changes.

**What remains unresolved.** Table 2 reports 7,056 raw fungal rows versus 7,089 in our saved file inspection, and 41,123 raw stability rows versus 65,356 locally. Different releases, other preprocessing, or reporting differences could explain this. Matching the retained totals does not prove that the files contain identical retained examples.

Truncation alone preserves row counts, so it does not explain the paper's reductions to 3,138 fungal and 23,929 stability examples. Additional selection or an incomplete description remains possible. Ab1 points to a further unresolved case: the paper lists a maximum of 1,203 nucleotides yet retains all 723 rows. Obtain the dataset and author preprocessing details before extending our four-benchmark rule to it.

**Consequence for the reproduction.** Retain the current rule as a documented implementation choice and report the actual rows used. Truncation can remove sequence context and requires an explicit policy for matching streams; it does not inevitably break correspondence if all streams are handled consistently. These uncertainties do not disappear because a score or row count matches.

---

## Published splits, and how much the test split reuses

Matching dataset size does not establish an independent evaluation. A **split** assigns examples to training, validation, or testing: training fits parameters, validation guides choices, and testing evaluates the resulting model. If an identical sequence appears on both sides, the test does not exclusively measure performance on new sequences.

The code retains the file's `Split` assignments after the current length filter and compares normalized sequence strings. It also groups repeated sequences to check whether their labels agree. This reports exact-sequence overlap, not evolutionary similarity or every possible source of dependence.

The output deliberately uses different counting units:

- `test rows seen in train` counts test observations, including repeated observations.
- `test sequences unseen` counts distinct test sequences absent from **both training and validation**.
- `sequences in >1 split` counts distinct sequence groups spread across split names.
- `repeats with differing labels` counts repeated sequence groups whose maximum label exceeds their minimum.

In [23]:
# Preserve the supplied split assignments and inspect exact normalized-sequence reuse.
rows = []
for key in BENCHMARKS:
    spec = datasets.DATASETS[key]
    frame = datasets.blf_benchmark_rows(key)
    split = frame[spec.split_column]
    # Sets count distinct sequences; the test frame below still counts rows.
    train_seqs = set(frame.loc[split == "train", "sequence"])
    val_seqs = set(frame.loc[split == "val", "sequence"])
    test = frame[split == "test"]
    test_seqs = set(test["sequence"])
    # Label spread is measured within each group of identical sequences.
    repeated = frame.groupby("sequence")[spec.label_column]
    sizes, spreads = repeated.size(), repeated.max() - repeated.min()
    rows.append({
        "benchmark": spec.label,
        "train": int((split == "train").sum()),
        "val": int((split == "val").sum()),
        "test": int((split == "test").sum()),
        "distinct sequences": int(frame["sequence"].nunique()),
        "sequences in >1 split": int((frame.groupby("sequence")[spec.split_column].nunique() > 1).sum()),
        "test rows seen in train": int(test["sequence"].isin(train_seqs).sum()),
        "test sequences unseen": len(test_seqs - train_seqs - val_seqs),
        "repeats with differing labels": int((spreads[sizes > 1] > 0).sum()),
    })
SPLITS = pd.DataFrame(rows).set_index("benchmark")
SPLITS

,train,val,test,distinct sequences,sequences in >1 split,test rows seen in train,test sequences unseen,repeats with differing labels
benchmark,,,,,,,,
Fungal expression,2153,555,430,3131,0,0,430,6
"E. coli protein expression (low, medium, high)",3010,728,712,4450,0,0,712,0
mRNA stability,16742,3608,3579,11303,3178,2444,985,4549
CoV-Vac degradation,1600,400,400,2400,0,0,400,0


**Saved findings.** Fungal expression, E. coli, and CoV-Vac show no exact sequence crossing split boundaries. That is narrower than calling these datasets “clean”: related sequences or other dependencies have not been assessed here, and fungal expression contains six repeated-sequence groups with differing labels within splits.

The mRNA stability data contain **23,929 rows but 11,303 distinct sequences**. Of the **3,579 test rows, 2,444** share a sequence with training:

$$
\text{test-row overlap} = \frac{2{,}444}{3{,}579} \approx 68.3\%.
$$

The table separately reports **985 distinct test sequences** absent from both training and validation. That is a different unit and comparison set; it is not the complement of the 2,444 overlapping test rows.

There are **4,549 repeated-sequence groups with differing labels** in the stability data. Assay conditions, biological context, or measurement variation are possible explanations, but this table does not establish the cause. Sequence identity does not necessarily imply identical measured targets.

**What this means for evaluation.** Overlap permits recognition of previously seen sequences and limits an unseen-sequence interpretation. It does not quantify how much a model's score comes from memorization, especially when repeated labels differ. The reproduction retains the published split for comparability. A separately reported sequence-grouped evaluation—keeping identical sequences on one side of the boundary—would address a different generalization question. It would still not exclude all homology or contextual dependence.

The earlier project protocol called for both published-split and grouped comparisons. No grouped evaluation or performance difference is computed here. The [Stage 1 published-split comparison](../stage1/Stage1_stability.ipynb#Published-split-comparison) examines the related issue on a smaller sample; its counts should not be substituted for these full-benchmark counts.

## Sequence checks

BioLangFusion's **codon grid** places streams at corresponding codon/amino-acid positions. A codon is a group of three nucleotides; translating those groups requires choosing a reading frame. Equal array lengths alone do not establish that the positions correspond biologically.

The previous checks concerned counts and evaluation. Here we inspect the retained sequences themselves: allowed RNA letters, the fraction with lengths divisible by three, and the fraction beginning with `AUG`. These are useful diagnostics, but none confirms the source sequence's annotated coding region or biological frame.

The last column asks what would happen if we applied Stage 1's [`is_retained`](../shared_code/datasets.py) rule. That helper requires a length divisible by three, an `ATG` or `AUG` start, and at least five translated residues before the first stop. It does **not** explicitly validate the alphabet; the table checks letters separately. Stage 3 does not apply that Stage 1 filter.

The proportion columns are displayed as fractions: `1.000` means 100%, and `0.016` means 1.6%.

In [24]:
# Diagnose retained inputs and the effect of the separate Stage 1 rule.
rows = []
for key in BENCHMARKS:
    spec = datasets.DATASETS[key]
    seq = datasets.blf_benchmark_rows(key)["sequence"]
    rows.append({
        "benchmark": spec.label,
        "kept rows": len(seq),
        "RNA letters only": bool(set("".join(seq)) <= set("ACGU")),
        "length a multiple of 3": f"{(seq.str.len() % 3 == 0).mean():.3f}",
        "starts with AUG": f"{(seq.str[:3] == 'AUG').mean():.3f}",
        "dropped by the Stage 1 filters": f"{1 - seq.map(datasets.is_retained).mean():.3f}",
    })
CHECKS = pd.DataFrame(rows).set_index("benchmark")
CHECKS

,kept rows,RNA letters only,length a multiple of 3,starts with AUG,dropped by the Stage 1 filters
benchmark,,,,,
Fungal expression,3138,True,1.000,0.932,0.068
"E. coli protein expression (low, medium, high)",4450,True,1.000,1.000,0.000
mRNA stability,23929,True,1.000,0.978,0.022
CoV-Vac degradation,2400,True,1.000,0.016,0.989


The saved checks show only RNA letters and lengths divisible by three in the retained rows. Nevertheless, only **1.6% of CoV-Vac rows begin with AUG**, and applying all Stage 1 filters would discard **98.9%**. Those percentages differ because the Stage 1 rule also tests translation length. Fungal expression would lose **6.8%**, and mRNA stability **2.2%**, under that combined rule. Applying it would substantially change the supplied benchmarks.

**Current protein-input choice.** This was originally left open for a later step, but it is now implemented in step 2 through [`grid_protein` and `inputs_for_rows`](../shared_code/fusion/token_embeddings.py). The helper translates complete triplets starting at the first supplied base, trims one or two leftover bases if present, and replaces stop symbols with `X`, an unspecified amino-acid placeholder. This preserves one protein-input position per complete triplet. It does not establish that a fragment begins in its biological reading frame or encodes a real protein with that sequence.

Apply the chosen protein-input policy consistently across encoder comparisons, so that changing the encoder does not also change which protein sequence the model receives.

Earlier alternatives were to translate from the first base, search for a longest open reading frame, or supply a missing protein modality. Those choices produce different model inputs. The implemented first-base, stop-to-`X` policy must be reported as our adaptation until its agreement with the authors' procedure is established; this audit does not select a new policy.

The [October 16 embedding run record](../../docs/run-records.md#2026-10-16-stage3-token-embeddings-token-level-caches-for-the-four-biolangfusion-benchmarks) records downstream extraction. A run record and a matched token count document execution and shape; they do not resolve the biological-frame question.

---

## Deviations recorded for Phase A

The audit identifies missing coverage, implementation choices, and interpretation limits. Recording them makes the comparison assessable; it does not, by itself, establish successful reproduction.

| Issue | Evidence or current state | Consequence for comparison |
| --- | --- | --- |
| Ab1 coverage | Not registered; separately sourced to Yazdani et al.; reported maximum 1,203 nt and 723 retained rows | State four-benchmark coverage until the dataset is obtained and verified. |
| Preprocessing | Our 999-nt row filter matches four used counts; the paper describes truncation, and raw counts disagree for two files | Report the applied rule and dataset identity. Keep author preprocessing and row identity unresolved. |
| Stability split overlap | 2,444 of 3,579 saved test rows reuse a training sequence; many repeated sequences have differing labels | Retain the published comparison with its limitation. Report any grouped evaluation separately, with its own producing run. |
| Protein input for fragments | Step 2 implements first-base triplet translation with stops represented as `X` | Identify it as an implemented adaptation; positional consistency is not proof of biological correspondence. |

Model construction adds questions this data audit cannot settle: TextCNN layer sizes, projection dimensions, loss/training details, and seeds. The earlier notebook recorded that no author implementation had been verified; this revision does not establish current public code availability. [Step 3](03_blf_build_check.ipynb) and the [run records](../../docs/run-records.md) hold implementation and execution evidence. Unspecified or conflicting paper details remain deviations to investigate, rather than reasons to assume a convenient choice is the original method.

### Earlier observations and planning notes

These notes preserve useful context from earlier versions without treating it as fresh evidence:

- The original notebook recorded an unsuccessful Ab1 search in the CodonBERT release and described the sequences as “heavy-chain.” That description needs verification against Ab1's original source; the search does not establish that the data are unavailable elsewhere. Ab1 acquisition was recorded as the first question for the October 20 meeting, and missing implementation details as the second.
- Repeated seeds and paired differences were linked to Overleaf Section 12.2, item 1; a published-versus-grouped stability comparison was also attributed to the manuscript protocol. These are retained planning references, not a fresh check of the manuscript or evidence that those comparisons ran.
- Earlier stability commentary reported 3,021 distinct test sequences, 4,579 repeated-sequence groups, and a median label spread of 0.94 on an approximately six-unit target range. The displayed split cell does not calculate all of these statistics, and they were not reverified in this revision. Its displayed values support the main interpretation above.
- The earlier “295-sequence gap” wording referred to Stage 1. Its saved comparison actually describes **295 sampled test rows** whose sequences are absent from the full published training split, not a grouped-split count or a measured 295-sequence performance gap.

Earlier explanations also treated count agreement as proof of a shared short-sequence subset, and justified 999 as one start token plus 999 nucleotides. Those inferences are not established by this audit. The revised preprocessing discussion retains the observations and the uncertainty separately.

---

## What step 2 needs from here

[Step 2](02_token_embeddings.ipynb) extracts a vector for each sequence token using the three frozen encoders. It consumes the shared loader's rows; this notebook does not export a separate selected dataset. The handoff requires:

1. **Row identity and order.** Keep the rows returned by `blf_benchmark_rows` in file order, with their labels, supplied splits, and original row indices attached. Each cached embedding row must refer to the same observation.
2. **Explicit encoder inputs.** Step 2 uses [`inputs_for_rows`](../shared_code/fusion/token_embeddings.py): nucleotide streams use the encoder's alphabet and protein inputs use `grid_protein`. This differs from Stage 1's `Dataset.inputs_for`/stop-at-first-stop path. Replacing U with T is an alphabet conversion, not an independently measured DNA sequence.
3. **Actual token-length checks.** A nucleotide cutoff is not a token budget. `token_states` checks the tokenizer's full input length against the configured encoder limit and raises if it would exceed it; framing tokens are removed from the cached representations. This is a downstream safeguard, not a check executed here.

For a 999-nucleotide sequence under the current helpers, the expected **content-token** counts are 999 for RNA-FM and 333 for the protein grid. Nucleotide Transformer uses 166 complete six-mers plus three single-nucleotide remainder tokens: **169**, not 167. Special tokens are additional model inputs and are handled separately. The current token helper owns these expectations; future tokenizer or encoder changes require rechecking them.

Token-level caches are much larger than Stage 1's pooled vectors. For RNA-FM alone, one maximum-length sequence contributes roughly $999 \times 640$ values before storage overhead. Use adequate local or cluster scratch storage; `Code/*/embeddings/` is ignored by Git. The shared result should retain the cache/run identity and settings through the existing [run records](../../docs/run-records.md), so later scores can be traced to the actual inputs.

The audit therefore provides an inspectable data contract and explicit unresolved questions. The next step can build representations from that contract; a faithful reproduction still depends on resolving or reporting the remaining data and implementation differences.